# 3. 系统增强（重构版）

这一节关注“系统级”问题：当单次检索流程已经优化过，仍需处理多轮记忆、多文档路由、工具编排与可恢复执行时，如何构建更稳健的 RAG 系统。

## 流程增强 vs 系统增强

流程增强关注"单次请求内"的多步决策（多轮检索、子问题拆解、质量把关）。

系统增强关注"跨请求"的工程问题：
- 上一轮对话的信息怎么延续？-> Memory
- 多个数据源怎么组织和路由？-> Multi-Document Agent
- 复杂任务怎么规划、执行、反思？-> Agentic RAG

判断标准：如果去掉 history / 去掉多文档路由 / 去掉任务规划，系统仍能回答，那是流程问题；否则是系统问题。


## Memory

### 无记忆失败场景
第二轮问题常包含“它 / 上面那个”这类代词；若系统不读历史，容易答错对象。

### 记忆增强思路
1. 保存最近 N 轮对话
2. 检索前拼接历史与当前问题
3. 回答后显式写回（write-back）

### 适用边界
- 适合：多轮问答、连续任务
- 局限：记忆过长会引入噪声，需做窗口与摘要控制

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma

load_dotenv()

MODEL_NAME = "gpt-4o-mini"
DATA_DIR = Path("notebook/C7 高级 RAG 技巧/6. 增强阶段/data/mutli_documents_data")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

class ConversationMemory:
    def __init__(self, max_turns=6):
        self.max_turns = max_turns
        self.turns = []

    def add_turn(self, role: str, text: str):
        self.turns.append((role, text.strip()))
        self.turns = self.turns[-self.max_turns:]

    def format(self) -> str:
        return "\n".join(f"{r}: {t}" for r, t in self.turns)

def build_query(user_query: str, memory: ConversationMemory | None) -> str:
    if memory is None or not memory.turns:
        return user_query
    return f"会话历史:\n{memory.format()}\n\n当前问题:\n{user_query}"

memory = ConversationMemory(max_turns=6)
memory.add_turn("user", "我先想看 Boston 的艺术文化特点")
memory.add_turn("assistant", "可以关注博物馆、音乐与社区活动")

turn2 = "那它和 Houston 比，文化活动有什么不同？"

prompt_wo_memory = f"仅根据当前问题回答：{turn2}"
answer_wo_memory = llm.invoke(prompt_wo_memory).content

query_with_memory = build_query(turn2, memory)
answer_with_memory = llm.invoke(query_with_memory).content

# 显式 write-back
memory.add_turn("user", turn2)
memory.add_turn("assistant", answer_with_memory)

print("=== 无记忆 ===")
print(answer_wo_memory[:180])
print("\n=== 有记忆 ===")
print(answer_with_memory[:180])

## Multi-Document Agent

### 单一索引失败场景
把所有城市文档混在一个索引里时，跨城市比较问题容易串信息。

### 系统化方案
1. 每个文档/城市单独建索引（相当于子 agent/tool）
2. 顶层路由先选工具，再做回答
3. 对照 baseline（单索引）观察差异

> 下方实现使用 LangChain 风格的“路由器 + 子检索器”，作为 LlamaIndex Agent 不可用时的等价教学实现。

In [ ]:
wiki_titles = ["Toronto", "Seattle", "Chicago", "Boston", "Houston"]

def build_city_retriever(city: str):
    loader = TextLoader(str(DATA_DIR / f"{city}.txt"), encoding="utf-8")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=280, chunk_overlap=40)
    chunks = splitter.split_documents(docs)
    vs = Chroma.from_documents(chunks, embedding=embeddings)
    return vs.as_retriever(search_kwargs={"k": 3})

city_retrievers = {city: build_city_retriever(city) for city in wiki_titles}

# baseline: 单索引（全部城市混合）
all_docs = []
for city in wiki_titles:
    all_docs.extend(TextLoader(str(DATA_DIR / f"{city}.txt"), encoding="utf-8").load())
all_chunks = RecursiveCharacterTextSplitter(chunk_size=280, chunk_overlap=40).split_documents(all_docs)
base_vs = Chroma.from_documents(all_chunks, embedding=embeddings)
base_retriever = base_vs.as_retriever(search_kwargs={"k": 4})

### 顶层路由（规则 + LLM）

- 规则路由：按城市关键词直接选子检索器
- LLM 路由：问题未显式提城市时，让模型判定优先工具

In [ ]:
def route_cities(query: str) -> list[str]:
    matched = [city for city in wiki_titles if city.lower() in query.lower()]
    if matched:
        return matched

    judge_prompt = f"""
从以下城市中选择与问题最相关的城市（可多选），用逗号分隔：{wiki_titles}
问题：{query}
只输出城市名，用逗号分隔。
"""
    raw = llm.invoke(judge_prompt).content.strip()
    cities = [c.strip() for c in raw.split(",") if c.strip() in wiki_titles]
    return cities if cities else ["Boston"]

def ask_multi_doc_agent(query: str) -> str:
    cities = route_cities(query)
    all_ctx = []
    for city in cities:
        docs = city_retrievers[city].invoke(query)
        all_ctx.extend(d.page_content for d in docs)
    ctx = "\n\n".join(all_ctx[:8])
    prompt = f"你是{'/'.join(cities)}领域助手。问题：{query}\n上下文：\n{ctx}\n请回答。"
    return cities, llm.invoke(prompt).content

response_cities, response = ask_multi_doc_agent("Tell me about the arts and culture in Boston")
print("route cities:", response_cities)
print(response[:260])

### 与单一索引 Baseline 对比

比较同一问题在两种方案中的输出：
- Baseline：混合索引直接检索
- Multi-Document Agent：先路由，再子检索

In [ ]:
test_query = "Tell me about the arts and culture in Boston"

agent_city, agent_response = ask_multi_doc_agent(test_query)
base_docs_ = base_retriever.invoke(test_query)
base_ctx = "\n\n".join(d.page_content for d in base_docs_)
baseline_response = llm.invoke(
    f"问题：{test_query}\n上下文：\n{base_ctx}\n请回答。"
).content

print("=== Agent Response ===")
print(f"route={agent_city}")
print(agent_response[:320])
print("\n=== Baseline Response ===")
print(baseline_response[:320])

In [ ]:
compare_query = "Tell the demographics of Houston, and then compare that with the demographics of Chicago"
agent_city2, agent_compare_response = ask_multi_doc_agent(compare_query)
base_docs2 = base_retriever.invoke(compare_query)
base_ctx2 = "\n\n".join(d.page_content for d in base_docs2)
baseline_compare_response = llm.invoke(
    f"问题：{compare_query}\n上下文：\n{base_ctx2}\n请回答。"
).content

print("=== Agent Compare Response ===")
print(f"route={agent_city2}")
print(agent_compare_response[:320])
print("\n=== Baseline Compare Response ===")
print(baseline_compare_response[:320])

## 系统成本与边界

- **延迟**：路由、工具调用、重试会增加端到端时延。
- **可观测性**：必须记录路由决策、检索上下文与中间输出。
- **权限控制**：不同工具/数据源应有最小权限策略。
- **回退策略**：路由失败或工具异常时，需回退到 baseline 或保守回答。

## Agentic RAG

### 核心结构
Agentic RAG 可以看成 Planner / Executor / Reflection 的循环：
1. Planner 拆解任务并决定先做什么
2. Executor 执行检索/路由/纠错等动作
3. Reflection 判断是否继续、修正还是结束

### 与非 Agent 流程对比
- 非 Agent：固定流程，路径稳定但灵活性有限
- Agentic：动态决策更强，但工程复杂度更高

In [ ]:
def planner(question: str) -> list[str]:
    plan_prompt = f"""
将问题拆成 2-3 个可独立检索的执行步骤。
每个步骤应是一个具体的检索问题，而非抽象描述。
只输出 Python 列表字符串。

问题：{question}
"""
    raw = llm.invoke(plan_prompt).content
    if '[' in raw and ']' in raw:
        try:
            return eval(raw)
        except Exception:
            pass
    return [f"查找关于 {question} 的关键事实", f"比较并生成关于 {question} 的结论"]

def executor(step: str) -> str:
    cities, ans = ask_multi_doc_agent(step)
    return f"[{step}] route={cities}\n{ans[:300]}"

def reflector(question: str, outputs: list[str]) -> tuple[str, list[str]]:
    review_prompt = f"""
原始问题：{question}
已有执行结果：
{chr(10).join(outputs)}

判断这些结果是否足够回答原始问题。
如果足够，输出：sufficient
如果不够，输出：insufficient: <需要补充检索的问题>
"""
    result = llm.invoke(review_prompt).content.strip()
    if "insufficient" in result.lower():
        extra = result.split(":")[-1].strip()
        return "insufficient", [extra] if extra else []
    return "sufficient", []

def agentic_rag(question: str, max_reflect: int = 2):
    steps = planner(question)
    outputs = [executor(step) for step in steps]

    for _ in range(max_reflect):
        decision, extra_steps = reflector(question, outputs)
        if decision == "sufficient" or not extra_steps:
            break
        for es in extra_steps:
            outputs.append(executor(es))
            steps.append(es)

    final = llm.invoke(f"问题：{question}\n中间结果：{outputs}\n请给出最终完整回答。").content
    return steps, outputs, final

agent_query = "Compare Boston and Houston in arts and demographics."
steps, outputs, agentic_answer = agentic_rag(agent_query)
non_agent_cities, non_agent_answer = ask_multi_doc_agent(agent_query)

print('steps:', steps)
print('\n=== Agentic RAG ===')
print(agentic_answer[:320])
print('\n=== Non-agent workflow ===')
print(f'route={non_agent_cities}')
print(non_agent_answer[:320])

### Agentic RAG 适用性与限制

- 适合：目标复杂、需要多步工具协作的场景。
- 限制：调试困难、成本高、对观测与回放能力要求高。
- 生产建议：先上线非 agent 路径作为回退，再逐步放量 agentic 策略。

## GraphRAG（轻量介绍）

### 向量检索 vs 图检索
- 向量检索：找"语义相似的文本片段"——适合局部事实查询
- 图检索：找"实体之间的关系路径"——适合全局性问题（"文档整体讲了什么"）和多跳推理（A→B→C）

### 核心流程
1. **实体与关系提取**：用 LLM 从文档中抽取 `(实体, 关系, 实体)` 三元组
2. **知识图谱构建**：将三元组存入图结构
3. **基于图的检索**：给定 query，先提取 query 中的实体，再在图中查找相关三元组作为上下文

### 适用场景
- 问题需要跨段落的实体关系推理（如"A 和 B 有什么联系"）
- 需要文档级全局摘要（如"这篇论文的核心贡献是什么"）
- 多文档间的实体对齐与关系发现

### 局限
- 实体提取质量依赖 LLM，噪声较大
- 图构建成本高（需要遍历所有文档）
- 生产环境通常需要 Neo4j 等图数据库，本节仅用 Python dict 做教学演示

### 与系统增强其他方法的关系
- Multi-Document Agent 按文档路由 -> GraphRAG 按实体关系路由
- Agentic RAG 做任务规划 -> GraphRAG 可作为其中一个工具

In [ ]:
# GraphRAG 最小教学示例：纯 dict 实现

from langchain_community.document_loaders import TextLoader

city_text = TextLoader(str(DATA_DIR / "Boston.txt"), encoding="utf-8").load()[0].page_content[:2000]

# Step 1: 用 LLM 提取三元组
extract_prompt = f"""
从以下文本中提取实体和关系，输出为 Python 列表格式：
[("实体1", "关系", "实体2"), ...]

要求：
- 每个三元组都是 (主语, 谓语/关系, 宾语)
- 提取 5-10 个最重要的三元组
- 只输出列表，不要其他内容

文本：
{city_text[:1500]}
"""

raw_triples = llm.invoke(extract_prompt).content
try:
    triples = eval(raw_triples)
except Exception:
    triples = [("Boston", "is_a", "city"), ("Boston", "known_for", "education")]

print(f"提取到 {len(triples)} 个三元组：")
for t in triples[:5]:
    print(f"  {t}")

# Step 2: 构建图（纯 dict）
graph = {}
for subj, rel, obj in triples:
    graph.setdefault(subj, []).append((rel, obj))
    graph.setdefault(obj, []).append((rel + "_by", subj))

# Step 3: 基于图的检索
def graph_retrieve(query: str, graph: dict, top_k: int = 5) -> str:
    entity_prompt = f"从问题中提取关键实体，用逗号分隔，只输出实体名。问题：{query}"
    entities = [e.strip() for e in llm.invoke(entity_prompt).content.split(",")]

    relevant_triples = []
    for entity in entities:
        for node, edges in graph.items():
            if entity.lower() in node.lower():
                for rel, target in edges:
                    relevant_triples.append(f"{node} --{rel}--> {target}")

    return "\n".join(relevant_triples[:top_k]) if relevant_triples else "未找到相关实体关系"

# 测试
test_q = "What is Boston known for in education?"
graph_context = graph_retrieve(test_q, graph)
print(f"\n图检索结果：\n{graph_context}")

graph_answer = llm.invoke(f"问题：{test_q}\n图谱证据：\n{graph_context}\n请回答。").content
print(f"\nGraphRAG 回答：\n{graph_answer[:300]}")

# 与向量检索对比
vector_docs = city_retrievers["Boston"].invoke(test_q)
vector_ctx = "\n\n".join(d.page_content for d in vector_docs)
vector_answer = llm.invoke(f"问题：{test_q}\n上下文：\n{vector_ctx}\n请回答。").content
print(f"\n向量检索回答：\n{vector_answer[:300]}")